In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp02
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                module_dir = self._local_data / module
                local_path = module_dir / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
                # Bundled data is present for this module but the file is absent
                # → it is genuinely missing. Raise immediately instead of falling
                # back to a Drive download that would hang offline (and pulls the
                # whole folder online just to discover the file is not there).
                if module_dir.is_dir():
                    raise FileNotFoundError(
                        f"File not found: {module}/{filename} "
                        f"(not in local data/{module}/)."
                    )
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp02/ex_5.py ──
"""
Shared infrastructure for MLFP02 Exercise 5 — Linear Regression.

Contains: HDB resale data loading, feature engineering, OLS fitting utilities,
diagnostic helpers, and visualisation helpers. Technique-specific code (the
derivation walk-throughs, the WLS weight construction, the polynomial/dummy
matrices) lives in the per-technique files, not here.
"""

from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats


# ════════════════════════════════════════════════════════════════════════
# CONSTANTS
# ════════════════════════════════════════════════════════════════════════

NUMERIC_FEATURES: list[str] = [
    "floor_area_sqm",
    "storey_midpoint",
    "remaining_lease_years",
]
TARGET: str = "resale_price"
BASE_FLAT_TYPE: str = "3 ROOM"

OUTPUT_DIR = Path("outputs") / "ex5_linear_regression"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — HDB resale flat transactions
# ════════════════════════════════════════════════════════════════════════


def load_hdb_clean() -> pl.DataFrame:
    """Load HDB resale data, engineer numeric features, drop nulls.

    Returns a polars DataFrame with columns:
      - floor_area_sqm (Float)
      - storey_midpoint (Float, midpoint of '07 TO 09' range)
      - remaining_lease_years (Float, 99 - years_elapsed_since_commence)
      - resale_price (target, SGD)
      - flat_type (categorical)
      - town (categorical)
    """
    loader = MLFPDataLoader()
    hdb = loader.load("mlfp01", "hdb_resale.parquet")

    hdb = hdb.with_columns(
        pl.col("month").str.to_date("%Y-%m").alias("transaction_date"),
    )
    hdb_recent = hdb.filter(pl.col("transaction_date") >= pl.date(2020, 1, 1))

    hdb_recent = hdb_recent.with_columns(
        (
            (
                pl.col("storey_range").str.extract(r"(\d+)", 1).cast(pl.Float64)
                + pl.col("storey_range").str.extract(r"TO (\d+)", 1).cast(pl.Float64)
            )
            / 2
        ).alias("storey_midpoint"),
        (99 - (pl.col("transaction_date").dt.year() - pl.col("lease_commence_date")))
        .cast(pl.Float64)
        .alias("remaining_lease_years"),
    )

    return hdb_recent.drop_nulls(
        subset=[*NUMERIC_FEATURES, TARGET],
    )


def build_design_matrix(
    df: pl.DataFrame, features: list[str] | None = None
) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """Build (X_with_intercept, y, feature_names) from a cleaned HDB frame."""
    features = features or NUMERIC_FEATURES
    y = df[TARGET].to_numpy().astype(np.float64)
    X_raw = df.select(features).to_numpy().astype(np.float64)
    n_obs = X_raw.shape[0]
    X = np.column_stack([np.ones(n_obs), X_raw])
    feature_names = ["intercept"] + list(features)
    return X, y, feature_names


# ════════════════════════════════════════════════════════════════════════
# OLS CORE — the workhorse every technique reuses
# ════════════════════════════════════════════════════════════════════════


def fit_ols(X: np.ndarray, y: np.ndarray) -> dict[str, Any]:
    """Fit OLS via the normal equation and return core statistics.

    Returns a dict with keys: beta, y_hat, residuals, XtX_inv, SSR, SST, SSE,
    R2, adj_R2, sigma_hat, se_beta, t_stats, p_values, f_stat, f_p_value, n, k.
    """
    n, k = X.shape
    XtX = X.T @ X
    XtX_inv = np.linalg.inv(XtX)
    beta = XtX_inv @ X.T @ y

    y_hat = X @ beta
    residuals = y - y_hat

    SSR = float(np.sum(residuals**2))
    SST = float(np.sum((y - y.mean()) ** 2))
    SSE = float(np.sum((y_hat - y.mean()) ** 2))

    sigma_sq = SSR / (n - k)
    sigma_hat = float(np.sqrt(sigma_sq))
    se_beta = np.sqrt(sigma_sq * np.diag(XtX_inv))
    t_stats = beta / se_beta
    # sf (survival function) keeps precision in the far tail; 1 - cdf
    # rounds to exactly 0 once the tail drops below ~1e-16.
    p_values = 2 * stats.t.sf(np.abs(t_stats), df=n - k)

    r2 = 1 - SSR / SST
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - k)
    f_stat = (SSE / (k - 1)) / (SSR / (n - k))
    f_p = stats.f.sf(f_stat, dfn=k - 1, dfd=n - k)

    return {
        "beta": beta,
        "y_hat": y_hat,
        "residuals": residuals,
        "XtX_inv": XtX_inv,
        "SSR": SSR,
        "SST": SST,
        "SSE": SSE,
        "R2": float(r2),
        "adj_R2": float(adj_r2),
        "sigma_hat": sigma_hat,
        "se_beta": se_beta,
        "t_stats": t_stats,
        "p_values": p_values,
        "f_stat": float(f_stat),
        "f_p_value": float(f_p),
        "n": int(n),
        "k": int(k),
    }


def format_p_value(p: float) -> str:
    """Render a p-value for printing, including the underflow case.

    Returns e.g. "= 3.20e-05", or "< 1e-300" when the tail probability is
    smaller than the smallest representable double and evaluates to 0.
    """
    return f"= {p:.2e}" if p > 0 else "< 1e-300"


# ════════════════════════════════════════════════════════════════════════
# K-FOLD CROSS-VALIDATION — fold assignment, from scratch
# ════════════════════════════════════════════════════════════════════════


def kfold_indices(
    n: int, k: int = 5, seed: int = 42
) -> list[tuple[np.ndarray, np.ndarray]]:
    """Return k (train_idx, test_idx) pairs from a seeded shuffle.

    Fold sizes differ by at most one row. The shuffle makes folds
    exchangeable — required because the HDB frame is sorted by month and
    an unshuffled split would make the last fold a different time period
    (that is a DIFFERENT technique: out-of-time validation, ex_8).
    """
    rng = np.random.default_rng(seed)
    order = rng.permutation(n)
    folds = np.array_split(order, k)
    pairs: list[tuple[np.ndarray, np.ndarray]] = []
    for i in range(k):
        test_idx = folds[i]
        train_idx = np.concatenate([folds[j] for j in range(k) if j != i])
        pairs.append((train_idx, test_idx))
    return pairs


def ols_r2_on(
    X: np.ndarray, y: np.ndarray, train_idx: np.ndarray, test_idx: np.ndarray
) -> float:
    """Fit OLS on train_idx, return R² computed on test_idx (out-of-sample)."""
    beta = np.linalg.lstsq(X[train_idx], y[train_idx], rcond=None)[0]
    y_true = y[test_idx]
    y_pred = X[test_idx] @ beta
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - y_true.mean()) ** 2))
    return 1.0 - ss_res / ss_tot


# ════════════════════════════════════════════════════════════════════════
# GEO FEATURES — town centroids and haversine distance to the CBD
# ════════════════════════════════════════════════════════════════════════
#
# The resale file has no coordinates, so locations come from a lookup of
# approximate TOWN-CENTRE coordinates (public geographic facts; teaching
# proxy — a production pipeline would geocode block + street via OneMap).
# Distance is to Raffles Place, the centre of the Central Business
# District.

CBD_RAFFLES_PLACE: tuple[float, float] = (1.2844, 103.8510)

TOWN_CENTROIDS: dict[str, tuple[float, float]] = {
    "ANG MO KIO": (1.3691, 103.8454),
    "BEDOK": (1.3236, 103.9273),
    "BISHAN": (1.3526, 103.8352),
    "BOON LAY": (1.3366, 103.7039),
    "BUKIT BATOK": (1.3490, 103.7496),
    "BUKIT MERAH": (1.2819, 103.8239),
    "BUKIT PANJANG": (1.3774, 103.7719),
    "BUKIT TIMAH": (1.3294, 103.8021),
    "CENTRAL AREA": (1.2903, 103.8520),
    "CHOA CHU KANG": (1.3840, 103.7470),
    "CLEMENTI": (1.3162, 103.7649),
    "GEYLANG": (1.3201, 103.8871),
    "HOUGANG": (1.3612, 103.8863),
    "JURONG EAST": (1.3329, 103.7436),
    "JURONG WEST": (1.3404, 103.7090),
    "KALLANG/WHAMPOA": (1.3100, 103.8651),
    "MARINE PARADE": (1.3017, 103.9057),
    "PASIR RIS": (1.3721, 103.9474),
    "PUNGGOL": (1.3984, 103.9072),
    "QUEENSTOWN": (1.2942, 103.7861),
    "SEMBAWANG": (1.4491, 103.8185),
    "SENGKANG": (1.3868, 103.8914),
    "SERANGOON": (1.3554, 103.8679),
    "TAMPINES": (1.3496, 103.9568),
    "TOA PAYOH": (1.3343, 103.8563),
    "WOODLANDS": (1.4382, 103.7890),
    "YISHUN": (1.4304, 103.8354),
}


def haversine_km(
    lat1: np.ndarray, lon1: np.ndarray, lat2: float, lon2: float
) -> np.ndarray:
    """Great-circle distance in km (mean Earth radius 6371 km)."""
    r = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp = np.radians(lat2 - lat1)
    dl = np.radians(lon2 - lon1)
    a = np.sin(dp / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2.0 * r * np.arcsin(np.sqrt(a))


def add_geo_features(df: pl.DataFrame) -> pl.DataFrame:
    """Add town centroid lat/lon and haversine distance-to-CBD (km).

    Rows whose town is missing from the centroid lookup become null and
    are dropped by the model frame's drop_nulls downstream.
    """
    towns = pl.DataFrame(
        {
            "town": list(TOWN_CENTROIDS.keys()),
            "town_lat": [c[0] for c in TOWN_CENTROIDS.values()],
            "town_lon": [c[1] for c in TOWN_CENTROIDS.values()],
        }
    )
    out = df.join(towns, on="town", how="left")
    lat = out["town_lat"].to_numpy().astype(np.float64)
    lon = out["town_lon"].to_numpy().astype(np.float64)
    dist = haversine_km(lat, lon, *CBD_RAFFLES_PLACE)
    return out.with_columns(
        pl.Series("dist_to_cbd_km", dist).cast(pl.Float64)
    )


# ════════════════════════════════════════════════════════════════════════
# EXPERIMENT TRACKING — kailash-ml ExperimentTracker
# ════════════════════════════════════════════════════════════════════════

TRACKER_STORE_URL = (
    f"sqlite:///{(OUTPUT_DIR / 'experiments.db').resolve().as_posix()}"
)


def track_train_run(
    experiment: str,
    run_name: str,
    params: dict[str, str],
    metrics: dict[str, float],
) -> str:
    """Log one Train-phase run to ExperimentTracker (sync wrapper).

    Returns the run_id. The tracker is closed in a finally block — kailash-ml
    holds the store connection open until close() is called.
    """
    import asyncio

    async def _log() -> str:
        from kailash_ml import ExperimentTracker

        tracker = await ExperimentTracker.create(store_url=TRACKER_STORE_URL)
        try:
            async with tracker.track(
                experiment=experiment, run_name=run_name
            ) as run:
                await run.log_params(params)
                await run.log_metrics(metrics)
                return run.run_id
        finally:
            await tracker.close()

    return asyncio.run(_log())


def print_coef_table(names: list[str], fit: dict[str, Any]) -> None:
    """Print coefficient / SE / t / p table for an OLS fit."""
    beta = fit["beta"]
    se = fit["se_beta"]
    t = fit["t_stats"]
    p = fit["p_values"]
    print(f"\n{'Feature':<25} {'β':>14} {'SE(β)':>12} {'t':>8} {'p':>10} {'Sig':>4}")
    print("-" * 78)
    for i, name in enumerate(names):
        if p[i] < 0.001:
            sig = "***"
        elif p[i] < 0.01:
            sig = "**"
        elif p[i] < 0.05:
            sig = "*"
        else:
            sig = "ns"
        print(
            f"{name:<25} {beta[i]:>14,.2f} {se[i]:>12,.2f} "
            f"{t[i]:>8.2f} {(f'{p[i]:.2e}' if p[i] > 0 else '<1e-300'):>10} {sig:>4}"
        )


# ════════════════════════════════════════════════════════════════════════
# DIAGNOSTICS — VIF, Breusch-Pagan, residual shape
# ════════════════════════════════════════════════════════════════════════


def compute_vif(X_raw: np.ndarray, feature_names: list[str]) -> dict[str, float]:
    """Variance Inflation Factor for each feature (no intercept column)."""
    n = X_raw.shape[0]
    results: dict[str, float] = {}
    for j in range(X_raw.shape[1]):
        other = [i for i in range(X_raw.shape[1]) if i != j]
        Xo = np.column_stack([np.ones(n), X_raw[:, other]])
        yj = X_raw[:, j]
        beta_j = np.linalg.lstsq(Xo, yj, rcond=None)[0]
        yhat_j = Xo @ beta_j
        ss_res = np.sum((yj - yhat_j) ** 2)
        ss_tot = np.sum((yj - yj.mean()) ** 2)
        r2_j = 1 - ss_res / ss_tot
        results[feature_names[j]] = (
            float(1.0 / (1.0 - r2_j)) if r2_j < 1 else float("inf")
        )
    return results


def breusch_pagan(residuals: np.ndarray, X_raw: np.ndarray) -> tuple[float, float]:
    """Breusch-Pagan test for heteroscedasticity. Returns (BP statistic, p-value)."""
    n = X_raw.shape[0]
    e_sq = residuals**2
    Xbp = np.column_stack([np.ones(n), X_raw])
    beta = np.linalg.lstsq(Xbp, e_sq, rcond=None)[0]
    pred = Xbp @ beta
    sse = np.sum((e_sq - pred) ** 2)
    sst = np.sum((e_sq - e_sq.mean()) ** 2)
    r2 = 1 - sse / sst
    bp_stat = n * r2
    p = stats.chi2.sf(bp_stat, df=X_raw.shape[1])
    return float(bp_stat), float(p)


# ════════════════════════════════════════════════════════════════════════
# VISUALISE — residual plots + actual-vs-predicted
# ════════════════════════════════════════════════════════════════════════


def save_residual_diagnostics(
    y_hat: np.ndarray,
    residuals: np.ndarray,
    feature_col: np.ndarray,
    feature_label: str,
    filename: str,
) -> Path:
    """Four-panel diagnostic figure: residuals vs fitted, histogram, Q-Q, vs feature."""
    fig = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=[
            "Residuals vs Fitted",
            "Residual Histogram",
            "Q-Q Plot",
            f"Residuals vs {feature_label}",
        ],
    )
    sample = min(3000, len(residuals))
    fig.add_trace(
        go.Scatter(
            x=y_hat[:sample],
            y=residuals[:sample],
            mode="markers",
            marker={"size": 2, "opacity": 0.3},
            name="Residuals",
        ),
        row=1,
        col=1,
    )
    fig.add_hline(y=0, row=1, col=1, line_dash="dash")
    fig.add_trace(go.Histogram(x=residuals, nbinsx=50, name="Residuals"), row=1, col=2)
    sorted_resid = np.sort(residuals)
    step = max(1, len(sorted_resid) // 2000)
    theoretical = stats.norm.ppf(np.linspace(0.001, 0.999, len(sorted_resid)))
    fig.add_trace(
        go.Scatter(
            x=theoretical[::step],
            y=sorted_resid[::step],
            mode="markers",
            marker={"size": 2},
            name="Q-Q",
        ),
        row=2,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=feature_col[:sample],
            y=residuals[:sample],
            mode="markers",
            marker={"size": 2, "opacity": 0.3},
            name=f"vs {feature_label}",
        ),
        row=2,
        col=2,
    )
    fig.update_layout(height=600, title="Residual Diagnostics", showlegend=False)
    path = OUTPUT_DIR / filename
    fig.write_html(str(path))
    return path


def save_actual_vs_predicted(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    title: str,
    filename: str,
) -> Path:
    """Actual-vs-predicted scatter with the perfect-prediction diagonal."""
    sample = min(2000, len(y_true))
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=y_true[:sample],
            y=y_pred[:sample],
            mode="markers",
            marker={"size": 3, "opacity": 0.4},
            name="Predictions",
        )
    )
    lo, hi = float(y_true.min()), float(y_true.max())
    fig.add_trace(
        go.Scatter(
            x=[lo, hi],
            y=[lo, hi],
            mode="lines",
            line={"dash": "dash", "color": "red"},
            name="Perfect",
        )
    )
    fig.update_layout(
        title=title,
        xaxis_title="Actual Price (SGD)",
        yaxis_title="Predicted Price (SGD)",
        height=500,
    )
    path = OUTPUT_DIR / filename
    fig.write_html(str(path))
    return path




# ════════════════════════════════════════════════════════════════════════
# MLFP02 — Exercise 5.4: Model Enrichment and Evaluation
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Extend models with polynomial and interaction terms
#   - Apply dummy variable encoding with a base category
#   - Cross-validate with train/test split and compute out-of-sample R-squared
#   - Compare model complexity: simple vs enriched vs categorical
#
# PREREQUISITES: Exercise 5.1-5.3 (OLS, diagnostics, WLS)
# ESTIMATED TIME: ~45 min
#
# TASKS:
#   1. Load data and fit baseline OLS
#   2. Add polynomial and interaction terms
#   3. Dummy variable encoding for flat type
#   4. Train/test split: out-of-sample evaluation
#   5. Model comparison and business interpretation
#
# THEORY:
#   Linear regression is "linear in parameters" — you can add x^2,
#   x1*x2, or dummy variables and still use OLS. The question is
#   whether the added complexity improves out-of-sample prediction
#   or just overfits the training data.
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import polars as pl
import plotly.graph_objects as go
from scipy import stats



## TASK 1 — Load Data and Fit Baseline


In [ ]:
print("=" * 70)
print("  MLFP02 Exercise 5.4: Model Enrichment and Evaluation")
print("=" * 70)

hdb_clean = load_hdb_clean()
X, y, feature_names = build_design_matrix(hdb_clean)
n_obs, k = X.shape
X_raw = X[:, 1:]  # Without intercept

fit_baseline = fit_ols(X, y)
r_squared = fit_baseline["R2"]
adj_r_squared = fit_baseline["adj_R2"]
SSR = fit_baseline["SSR"]
SST = fit_baseline["SST"]

print(f"\n  Baseline OLS: R-squared={r_squared:.6f}, Adj R-squared={adj_r_squared:.6f}")



## THEORY — Beyond Straight Lines


In [ ]:
# "Linear regression" is linear in PARAMETERS, not in features.
# You can transform features any way you like — square them, multiply
# them, take logs — and still use the normal equation. The key
# question: does the extra complexity capture real patterns or just
# noise?
#
# Analogy: Imagine fitting a price model for HDB flats. A straight
# line says "each extra sqm adds $X." But reality might be: small
# flats have a premium per sqm (scarcity), large flats have a
# premium per sqm (luxury). A quadratic term captures this curve.
# An interaction term says "the storey premium is bigger for large
# flats" — a penthouse effect.
#
# WHY THIS MATTERS: A property developer evaluating whether to build
# 100 small flats or 50 large ones needs to know whether the
# price-per-sqm curve is linear, convex, or concave. The polynomial
# and interaction terms answer this directly.



## TASK 2 — Polynomial and Interaction Terms


In [ ]:
# Non-linearity: floor_area^2 captures diminishing/increasing returns
# Interactions: storey * area captures "premium for high-floor large flats"

print(f"\n=== Polynomial and Interaction Terms ===")

area = X_raw[:, 0]
storey = X_raw[:, 1]
lease = X_raw[:, 2]

X_enriched = np.column_stack(
    [
        np.ones(n_obs),
        area,
        storey,
        lease,
        area**2,  # Polynomial: diminishing returns on area
        storey * area,  # Interaction: high-floor premium x size
        lease * area,  # Interaction: lease x size
    ]
)
enriched_names = [
    "intercept",
    "area",
    "storey",
    "lease",
    "area_sq",
    "storey_x_area",
    "lease_x_area",
]
k_enriched = X_enriched.shape[1]

# Fit enriched model
beta_enriched = np.linalg.lstsq(X_enriched, y, rcond=None)[0]
y_hat_enriched = X_enriched @ beta_enriched
resid_enriched = y - y_hat_enriched
ssr_enriched = float(np.sum(resid_enriched**2))
r2_enriched = 1 - ssr_enriched / SST
adj_r2_enriched = 1 - (1 - r2_enriched) * (n_obs - 1) / (n_obs - k_enriched)

# F-test: enriched vs simple model
f_improvement = ((SSR - ssr_enriched) / (k_enriched - k)) / (
    ssr_enriched / (n_obs - k_enriched)
)
f_p_improvement = stats.f.sf(f_improvement, dfn=k_enriched - k, dfd=n_obs - k_enriched)

print(f"{'Feature':<20} {'Coefficient':>14}")
print("-" * 38)
for name, coef in zip(enriched_names, beta_enriched):
    print(f"{name:<20} {coef:>14,.4f}")

print(f"\nSimple model:   R-squared={r_squared:.6f}, Adj R-squared={adj_r_squared:.6f}")
print(
    f"Enriched model: R-squared={r2_enriched:.6f}, Adj R-squared={adj_r2_enriched:.6f}"
)
print(
    f"F-test (enriched vs simple): F={f_improvement:.2f}, "
    f"p {format_p_value(f_p_improvement)}"
)
print(
    f"Enriched model is "
    f"{'significantly better' if f_p_improvement < 0.05 else 'NOT significantly better'}"
)

# INTERPRETATION: The area^2 term captures non-linearity — perhaps
# price per sqm increases for very large flats (premium penthouses)
# or decreases (diminishing returns). The interaction storey*area
# captures whether the storey premium is larger for bigger flats.



### Checkpoint 2


In [ ]:
assert (
    r2_enriched >= r_squared - 0.001
), "Adding features should not decrease R-squared substantially"
print("\n--- Checkpoint 2 passed --- enriched model built\n")



## TASK 3 — Dummy Variable Encoding


In [ ]:
# Categorical variables -> binary dummies. Drop one category to avoid
# the dummy variable trap (perfect multicollinearity with intercept).

print(f"\n=== Dummy Variable Encoding ===")

flat_types_in_data = sorted(hdb_clean["flat_type"].unique().to_list())
print(f"Flat types: {flat_types_in_data}")

# BASE_FLAT_TYPE (3 ROOM) is the base: a familiar reference point that
# makes each coefficient read as "premium over a 3-room flat". It is NOT
# the most common type (4 ROOM is) — any category works as the base; the
# choice only changes what the coefficients are measured against.
type_counts = hdb_clean["flat_type"].value_counts().sort("count", descending=True)
print(f"Most common type: {type_counts['flat_type'][0]} ({type_counts['count'][0]:,} sales)")
dummy_categories = [ft for ft in flat_types_in_data if ft != BASE_FLAT_TYPE]

# Build dummy columns
dummy_arrays = []
for ft in dummy_categories:
    dummy = (hdb_clean["flat_type"].to_numpy() == ft).astype(np.float64)
    dummy_arrays.append(dummy)

X_with_dummies = np.column_stack(
    [
        np.ones(n_obs),
        X_raw,  # Original numeric features
        np.column_stack(dummy_arrays),  # Dummy variables
    ]
)
dummy_names = (
    ["intercept"]
    + list(NUMERIC_FEATURES)
    + [f"flat_{ft.replace(' ', '_')}" for ft in dummy_categories]
)
k_dummy = X_with_dummies.shape[1]

# Fit model with dummies
beta_dummy = np.linalg.lstsq(X_with_dummies, y, rcond=None)[0]
y_hat_dummy = X_with_dummies @ beta_dummy
ssr_dummy = float(np.sum((y - y_hat_dummy) ** 2))
r2_dummy = 1 - ssr_dummy / SST
adj_r2_dummy = 1 - (1 - r2_dummy) * (n_obs - 1) / (n_obs - k_dummy)

print(f"\nBase category: {BASE_FLAT_TYPE}")
print(f"\n{'Feature':<30} {'Coefficient':>14}")
print("-" * 48)
for name, coef in zip(dummy_names, beta_dummy):
    print(f"{name:<30} {coef:>14,.0f}")

print(
    f"\nModel with dummies: R-squared={r2_dummy:.6f}, Adj R-squared={adj_r2_dummy:.6f}"
)
print(f"Improvement over simple: Delta R-squared={r2_dummy - r_squared:+.6f}")

# INTERPRETATION: Each dummy coefficient represents the price premium
# (or discount) relative to the base category (3 ROOM). For example,
# if the 5 ROOM coefficient is +$150K, then 5-room flats sell for
# $150K more than 3-room flats, all else equal.



### Checkpoint 3


In [ ]:
assert r2_dummy > r_squared, "Adding flat type should improve R-squared"
assert len(dummy_categories) == len(flat_types_in_data) - 1, "Should drop one category"
print("\n--- Checkpoint 3 passed --- dummy encoding completed\n")



## TASK 4 — Train/Test Split: Out-of-Sample Evaluation


In [ ]:
print(f"\n=== Train/Test Split ===")

rng = np.random.default_rng(seed=42)
n_total_obs = X_with_dummies.shape[0]
indices = rng.permutation(n_total_obs)
split_point = int(0.8 * n_total_obs)

train_idx = indices[:split_point]
test_idx = indices[split_point:]

X_train = X_with_dummies[train_idx]
y_train = y[train_idx]
X_test = X_with_dummies[test_idx]
y_test = y[test_idx]

# Fit on train
beta_train = np.linalg.lstsq(X_train, y_train, rcond=None)[0]

# Evaluate on both
y_train_pred = X_train @ beta_train
y_test_pred = X_test @ beta_train

r2_train = 1 - float(np.sum((y_train - y_train_pred) ** 2)) / float(
    np.sum((y_train - y_train.mean()) ** 2)
)
r2_test = 1 - float(np.sum((y_test - y_test_pred) ** 2)) / float(
    np.sum((y_test - y_test.mean()) ** 2)
)
rmse_train = float(np.sqrt(np.mean((y_train - y_train_pred) ** 2)))
rmse_test = float(np.sqrt(np.mean((y_test - y_test_pred) ** 2)))
mae_train = float(np.mean(np.abs(y_train - y_train_pred)))
mae_test = float(np.mean(np.abs(y_test - y_test_pred)))

print(f"Train: n={len(train_idx):,}")
print(f"Test:  n={len(test_idx):,}")
print(f"\n{'Metric':<12} {'Train':>14} {'Test':>14} {'Delta':>10}")
print("-" * 54)
print(
    f"{'R-squared':<12} {r2_train:>14.6f} {r2_test:>14.6f} {r2_test - r2_train:>+10.6f}"
)
print(
    f"{'RMSE':<12} ${rmse_train:>12,.0f} ${rmse_test:>12,.0f} "
    f"${rmse_test - rmse_train:>+8,.0f}"
)
print(
    f"{'MAE':<12} ${mae_train:>12,.0f} ${mae_test:>12,.0f} "
    f"${mae_test - mae_train:>+8,.0f}"
)

gap = abs(r2_train - r2_test)
print(f"\nTrain-test R-squared gap: {gap:.4f}")
if gap < 0.02:
    print("Minimal overfitting — model generalises well")
elif gap < 0.05:
    print("Slight overfitting — consider regularisation")
else:
    print("OVERFITTING — model is too complex for the data")

# INTERPRETATION: If train R-squared >> test R-squared, the model
# memorises training data instead of learning generalisable patterns.
# The train-test gap tells you whether your model complexity is
# appropriate.



### Checkpoint 4


In [ ]:
assert r2_test > 0, "Out-of-sample R-squared must be positive"
assert (
    r2_train >= r2_test - 0.05
), "Train R-squared should be >= test R-squared (approx)"
print("\n--- Checkpoint 4 passed --- out-of-sample evaluation completed\n")



## TASK 5 — Model Comparison Summary


In [ ]:
print(f"\n=== Model Comparison Summary ===")
print(f"{'Model':<30} {'R-sq':>10} {'Adj R-sq':>10} {'k':>4}")
print("-" * 58)
print(f"{'Simple (3 features)':<30} {r_squared:>10.6f} {adj_r_squared:>10.6f} {k:>4}")
print(
    f"{'Enriched (poly+interact)':<30} {r2_enriched:>10.6f} {adj_r2_enriched:>10.6f} "
    f"{k_enriched:>4}"
)
print(
    f"{'With flat type dummies':<30} {r2_dummy:>10.6f} {adj_r2_dummy:>10.6f} {k_dummy:>4}"
)



### Checkpoint 5


In [ ]:
print("\n--- Checkpoint 5 passed --- model comparison complete\n")



## VISUALISE — Test Set Actual vs Predicted


In [ ]:
path = save_actual_vs_predicted(
    y_test,
    y_test_pred,
    title="Full Model: Actual vs Predicted (Test Set)",
    filename="04_test_actual_vs_predicted.html",
)
print(f"Saved: {path}")

# --- Polynomial fit curve: price vs floor area with linear + quadratic ---
# Both curves hold storey and lease at their medians, so the only
# difference between them is the shape in area — a like-for-like view.
area_sorted_idx = np.argsort(area)
area_sorted = area[area_sorted_idx]
med_storey = float(np.median(storey))
med_lease = float(np.median(lease))
# Linear model: intercept + beta_area * area + storey/lease at medians
beta_baseline = fit_baseline["beta"]
y_linear = (
    beta_baseline[0]
    + beta_baseline[1] * area_sorted
    + beta_baseline[2] * med_storey
    + beta_baseline[3] * med_lease
)
# Enriched model at the same storey/lease medians
y_poly = (
    beta_enriched[0]
    + beta_enriched[1] * area_sorted
    + beta_enriched[2] * med_storey
    + beta_enriched[3] * med_lease
    + beta_enriched[4] * area_sorted**2
    + beta_enriched[5] * med_storey * area_sorted
    + beta_enriched[6] * med_lease * area_sorted
)

sample = min(3000, n_obs)
fig_poly = go.Figure()
fig_poly.add_trace(
    go.Scatter(
        x=area[:sample].tolist(),
        y=y[:sample].tolist(),
        mode="markers",
        marker={"size": 2, "opacity": 0.2, "color": "#94A3B8"},
        name="Data",
    )
)
fig_poly.add_trace(
    go.Scatter(
        x=area_sorted.tolist(),
        y=y_linear.tolist(),
        mode="lines",
        line={"color": "#2563EB", "width": 2},
        name="Linear",
    )
)
fig_poly.add_trace(
    go.Scatter(
        x=area_sorted.tolist(),
        y=y_poly.tolist(),
        mode="lines",
        line={"color": "#DC2626", "width": 2, "dash": "dash"},
        name="Polynomial + Interactions",
    )
)
fig_poly.update_layout(
    title="Price vs Floor Area — Does a Curve Fit Better Than a Line?",
    xaxis_title="Floor Area (sqm)",
    yaxis_title="Predicted Price (SGD)",
    height=450,
)
path_poly = OUTPUT_DIR / "04_polynomial_fit_curves.html"
fig_poly.write_html(str(path_poly))
print(f"Saved: {path_poly}")

# --- Train/test comparison bar chart: R-squared, RMSE, MAE ---
metrics = ["R-squared", "RMSE ($K)", "MAE ($K)"]
train_vals = [r2_train, rmse_train / 1000, mae_train / 1000]
test_vals = [r2_test, rmse_test / 1000, mae_test / 1000]

fig_tt = go.Figure()
fig_tt.add_trace(
    go.Bar(
        x=metrics,
        y=train_vals,
        name="Train",
        marker_color="#2563EB",
    )
)
fig_tt.add_trace(
    go.Bar(
        x=metrics,
        y=test_vals,
        name="Test",
        marker_color="#DC2626",
    )
)
fig_tt.update_layout(
    title="Train vs Test Performance — Is the Model Overfitting?",
    yaxis_title="Value",
    barmode="group",
    height=400,
)
path_tt = OUTPUT_DIR / "04_train_test_comparison.html"
fig_tt.write_html(str(path_tt))
print(f"Saved: {path_tt}")



## APPLY — Property Developer Unit-Mix Decision


Predicted price from the dummy model at median storey and lease.


In [ ]:
# SCENARIO: A Singapore developer is choosing the unit mix for a site,
# using HDB resale prices as a demand proxy. Option A: 100 small units
# (67 sqm, 3-room-like). Option B: 50 large units (110 sqm, 5-room-like).
# Construction cost is similar per sqm.
#
# The dummy-encoded model prices each unit type at the median storey
# and lease: base price + area effect + the flat-type premium. The
# out-of-sample RMSE says how far a single unit's price can miss.

dummy_index = {name: i for i, name in enumerate(dummy_names)}


def price_unit(area_sqm: float, flat_type: str) -> float:
    x_new = np.zeros(k_dummy)
    x_new[0] = 1.0
    x_new[1:4] = [area_sqm, med_storey, med_lease]
    dummy_name = f"flat_{flat_type.replace(' ', '_')}"
    if dummy_name in dummy_index:  # the base category has no dummy
        x_new[dummy_index[dummy_name]] = 1.0
    return float(x_new @ beta_dummy)


print(f"\n--- Business Application: Developer Unit-Mix Decision ---")
unit_a = price_unit(67.0, "3 ROOM")
unit_b = price_unit(110.0, "5 ROOM")
revenue_a = 100 * unit_a
revenue_b = 50 * unit_b
print(f"  5-ROOM premium over 3-ROOM at equal area: ${beta_dummy[dummy_index['flat_5_ROOM']]:,.0f}")
print(f"  Option A: 100 x ${unit_a:,.0f} = ${revenue_a:,.0f}  (6,700 sqm)")
print(f"  Option B:  50 x ${unit_b:,.0f} = ${revenue_b:,.0f}  (5,500 sqm)")
print(
    f"  Revenue per sqm built: A ${revenue_a / 6_700:,.0f}  vs  B ${revenue_b / 5_500:,.0f}"
)
better = "A (many small units)" if revenue_a / 6_700 > revenue_b / 5_500 else "B (fewer large units)"
print(f"  On this model, option {better} earns more per sqm built.")

print(f"\n  Model reliability:")
print(f"  Train-test R-squared gap: {gap:.4f}")
print(f"  Out-of-sample RMSE: ${rmse_test:,.0f} per unit")
print(
    f"  The model explains only {r2_test:.0%} of out-of-sample price variation —\n"
    f"  town, MRT access and condition are missing, so treat the comparison as\n"
    f"  a first screen, not a valuation."
)



## REFLECTION


- Polynomial terms (area^2) and interactions (storey x area)
  - F-test for nested model comparison (simple vs enriched)
  - Dummy encoding with base category to avoid the dummy trap
  - Train/test split: out-of-sample R-squared, RMSE, MAE
  - Model complexity trade-off: more features != better prediction
  - Business reasoning: pricing a unit-mix decision with dummy coefficients


FULL EXERCISE SUMMARY:
  - 5.1: OLS from scratch, coefficient interpretation, significance
  - 5.2: Diagnostics — VIF, residual normality, Breusch-Pagan
  - 5.3: Weighted Least Squares for heteroscedastic data
  - 5.4: Model enrichment, dummy encoding, train/test evaluation

  NEXT: In Exercise 6 you'll build logistic regression for binary
  classification. You'll implement the sigmoid function, maximise
  the Bernoulli log-likelihood, interpret coefficients as odds ratios,
  and perform ANOVA for multi-group comparison.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED (5.4)")
print("=" * 70)
print(
)

print("=" * 70)
print("  EXERCISE 5 COMPLETE — Linear Regression")
print("=" * 70)
print(
)

